In [22]:
import torch
import os

class Config:
  n_decoder = 6
  d_model = 128
  context_length = 256
  max_len = 128
  batch_size = 8
  block_size = 128

In [3]:
from typing import Dict
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


class Config:
    # kiến trúc
    d_model = 128
    num_heads = 4
    num_layers = 4
    block_size = 128          # context length
    # training
    batch_size = 64
    lr = 3e-4
    epochs = 3
    log_every = 200


class Tokenizer:
    """Char-level, xây vocab ĐỘNG từ corpus thay vì hard-code 4 ký tự."""

    def __init__(self, corpus: str):
        chars = sorted(list(set(corpus)))
        # id 0 dành riêng cho ký tự lạ / không có trong vocab train
        self.stoi: Dict[str, int] = {ch: i + 1 for i, ch in enumerate(chars)}
        self.itos: Dict[int, str] = {i: ch for ch, i in self.stoi.items()}
        self.vocab_size = len(self.stoi) + 1  # +1 cho id 0

    def encode(self, text: str) -> torch.Tensor:
        ids = [self.stoi.get(ch, 0) for ch in text]
        return torch.tensor(ids, dtype=torch.long)

    def decode(self, indices: torch.Tensor) -> str:
        return "".join(self.itos.get(int(i), "") for i in indices.tolist())


class CharDataset(Dataset):
    """
    KHÔNG cắt trước hàng triệu substring ra RAM (sketch gốc làm vậy -> rất tốn bộ nhớ
    và mỗi item vẫn là string thô, chưa encode). Lưu 1 tensor id duy nhất, cắt lát
    on-the-fly trong __getitem__.
    """

    def __init__(self, data_ids: torch.Tensor, block_size: int):
        self.data = data_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size - 1

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.block_size]
        y = self.data[idx + 1: idx + self.block_size + 1]
        return x, y


class RMSNorm(nn.Module):
    """Tự viết thay vì phụ thuộc nn.RMSNorm (chỉ có từ torch bản khá mới)."""

    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return normed * self.weight


class Embedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, block_size: int):
        super().__init__()
        self.token_emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(num_embeddings=block_size, embedding_dim=d_model)

    def forward(self, index: torch.Tensor) -> torch.Tensor:
        B, T = index.shape
        positions = torch.arange(T, device=index.device)
        # (B,T,d_model) + (T,d_model) -> broadcast đúng theo batch
        return self.token_emb(index) + self.pos_emb(positions)


class Attention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, block_size: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model phải chia hết cho num_heads"
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(in_features=d_model, out_features=d_model * 3)
        self.out_proj = nn.Linear(in_features=d_model, out_features=d_model)  # sketch gốc THIẾU lớp này

        # causal mask cố định, không học -> đăng ký buffer để tự chuyển device theo model
        mask = torch.tril(torch.ones(block_size, block_size)).bool()
        self.register_buffer("causal_mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, d_model = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)  # mỗi cái (B, T, d_model)

        # tách head ĐÚNG cách: reshape rồi transpose, không reshape thẳng
        # (B,T,d_model) -> (B,T,num_heads,head_dim) -> (B,num_heads,T,head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B,num_heads,T,T)
        # causal mask: vị trí t không được nhìn thấy vị trí > t -> ĐÂY LÀ PHẦN SKETCH GỐC THIẾU HOÀN TOÀN
        attn_scores = attn_scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn_weights = torch.softmax(attn_scores, dim=-1)

        out = attn_weights @ v  # (B,num_heads,T,head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, d_model)  # gộp head lại đúng thứ tự
        return self.out_proj(out)


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, block_size: int):
        super().__init__()
        self.norm1 = RMSNorm(d_model)  # sketch gốc chỉ có 1 norm dùng chung 2 lần -> sai
        self.norm2 = RMSNorm(d_model)
        self.attention = Attention(d_model=d_model, num_heads=num_heads, block_size=block_size)

        hidden = d_model * 4  # sketch gốc không mở rộng chiều ẩn của FFN
        self.fnn1 = nn.Linear(in_features=d_model, out_features=hidden)
        self.fnn2 = nn.Linear(in_features=hidden, out_features=d_model)
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # residual connection: sketch gốc THIẾU hoàn toàn cả 2 residual này
        x = x + self.attention(self.norm1(x))
        x = x + self.fnn2(self.act(self.fnn1(self.norm2(x))))
        return x


class MyModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, num_layers: int, block_size: int):
        super().__init__()
        self.embedding = Embedding(vocab_size=vocab_size, d_model=d_model, block_size=block_size)
        self.decoder = nn.ModuleList([  # sketch gốc dùng nn.Sequential(list) -> lỗi, Sequential không nhận list
            DecoderBlock(d_model=d_model, num_heads=num_heads, block_size=block_size)
            for _ in range(num_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(in_features=d_model, out_features=vocab_size)  # sketch gốc để out_features=num_heads (sai)
        self.lm_head.weight = self.embedding.token_emb.weight  # weight tying (mẹo chuẩn của GPT-2/nanoGPT)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(input_ids)
        for block in self.decoder:
            x = block(x)
        x = self.final_norm(x)  # KHÔNG cắt out[-1] như sketch gốc (nó lấy nhầm sample cuối trong batch)
        return self.lm_head(x)   # (B, T, vocab_size) - logits ở MỌI vị trí, cần cho loss (B,T)


def train(model, dataloader, val_dataloader, criterion, optimizer, device):
    model.train()
    for epoch in range(Config.epochs):
        running_loss = 0.0
        for step, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()                                    # sketch gốc THIẾU dòng này
            logits = model(x)                                        # (B,T,vocab_size)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()                                          # sketch gốc THIẾU dòng này
            optimizer.step()                                         # sketch gốc THIẾU dòng này

            running_loss += loss.item()
            if step % Config.log_every == 0:
                print(f"epoch {epoch} step {step} loss {running_loss / (step + 1):.4f}")

        val_loss = evaluate(model, val_dataloader, criterion, device)
        print(f"== epoch {epoch} xong | train loss {running_loss/len(dataloader):.4f} | val loss {val_loss:.4f} ==")


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        total_loss += loss.item()
    model.train()
    return total_loss / len(dataloader)


# if __name__ == "__main__":
#     device = "cuda" if torch.cuda.is_available() else "cpu"

#     train_text = open("train.txt", "r", encoding="utf-8").read()
#     val_text = open("val.txt", "r", encoding="utf-8").read()

#     tokenizer = Tokenizer(corpus=train_text)  # xây vocab từ train, val DÙNG CHUNG vocab này

#     train_ids = tokenizer.encode(train_text)
#     val_ids = tokenizer.encode(val_text)

#     train_dataset = CharDataset(train_ids, block_size=Config.block_size)
#     val_dataset = CharDataset(val_ids, block_size=Config.block_size)

#     dataloader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True, num_workers=2)
#     val_dataloader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False, num_workers=2)

#     model = MyModel(
#         vocab_size=tokenizer.vocab_size,
#         d_model=Config.d_model,
#         num_heads=Config.num_heads,
#         num_layers=Config.num_layers,
#         block_size=Config.block_size,
#     ).to(device)  # sketch gốc TẠO model xong không bao giờ .to(device)

#     optimizer = torch.optim.AdamW(model.parameters(), lr=Config.lr)  # sketch gốc gọi Adam() thiếu params, và tạo TRƯỚC model
#     criterion = nn.CrossEntropyLoss(ignore_index=0)                   # sketch gốc: criterion = torch (vô nghĩa)

#     train(model, dataloader, val_dataloader, criterion, optimizer, device)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

train_text = open("train.txt", "r", encoding="utf-8").read()
val_text = open("val.txt", "r", encoding="utf-8").read()

tokenizer = Tokenizer(corpus=train_text)  # xây vocab từ train, val DÙNG CHUNG vocab này

train_ids = tokenizer.encode(train_text)
val_ids = tokenizer.encode(val_text)

train_dataset = CharDataset(train_ids, block_size=Config.block_size)
val_dataset = CharDataset(val_ids, block_size=Config.block_size)

dataloader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False, num_workers=2)

model = MyModel(
    vocab_size=tokenizer.vocab_size,
    d_model=Config.d_model,
    num_heads=Config.num_heads,
    num_layers=Config.num_layers,
    block_size=Config.block_size,
).to(device)  # sketch gốc TẠO model xong không bao giờ .to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Config.lr)  # sketch gốc gọi Adam() thiếu params, và tạo TRƯỚC model
criterion = nn.CrossEntropyLoss(ignore_index=0)                   # sketch gốc: criterion = torch (vô nghĩa)

train(model, dataloader, val_dataloader, criterion, optimizer, device)

epoch 0 step 0 loss 86.5173
epoch 0 step 200 loss 9.2483
epoch 0 step 400 loss 6.0960
epoch 0 step 600 loss 4.9775
epoch 0 step 800 loss 4.3921
epoch 0 step 1000 loss 4.0270
epoch 0 step 1200 loss 3.7758
epoch 0 step 1400 loss 3.5909
epoch 0 step 1600 loss 3.4478
epoch 0 step 1800 loss 3.3313
epoch 0 step 2000 loss 3.2320
epoch 0 step 2200 loss 3.1451
epoch 0 step 2400 loss 3.0677
epoch 0 step 2600 loss 2.9978
epoch 0 step 2800 loss 2.9343
epoch 0 step 3000 loss 2.8758
epoch 0 step 3200 loss 2.8216
epoch 0 step 3400 loss 2.7715
epoch 0 step 3600 loss 2.7248
epoch 0 step 3800 loss 2.6814
epoch 0 step 4000 loss 2.6409
epoch 0 step 4200 loss 2.6024
epoch 0 step 4400 loss 2.5664
epoch 0 step 4600 loss 2.5321
epoch 0 step 4800 loss 2.4997
epoch 0 step 5000 loss 2.4691
epoch 0 step 5200 loss 2.4400
epoch 0 step 5400 loss 2.4122
epoch 0 step 5600 loss 2.3856
epoch 0 step 5800 loss 2.3606
epoch 0 step 6000 loss 2.3367
epoch 0 step 6200 loss 2.3137
epoch 0 step 6400 loss 2.2918
epoch 0 step 660

In [5]:

@torch.no_grad()
def generate(model, tokenizer: Tokenizer, prompt: str, max_new_tokens: int, device,
             temperature: float = 1.0, top_p: float = 0.9) -> str:
    """
    Sinh văn bản autoregressive từ 1 prompt.
    - temperature: chia logits trước softmax. <1 -> phân phối "nhọn" hơn (ít ngẫu nhiên),
      >1 -> "phẳng" hơn (sáng tạo/loạn hơn).
    - top_p (nucleus sampling): chỉ giữ tập token nhỏ nhất có tổng xác suất >= top_p,
      cắt bỏ phần đuôi xác suất thấp trước khi sample.
    """
    model.eval()
    ids = tokenizer.encode(prompt).unsqueeze(0).to(device)  # (1, T)

    for _ in range(max_new_tokens):
        # positional embedding chỉ học tới block_size vị trí -> phải crop context nếu chuỗi dài hơn
        ids_cond = ids[:, -Config.block_size:]
        logits = model(ids_cond)                                    # (1, T, vocab)
        logits = logits[:, -1, :] / max(temperature, 1e-6)          # chỉ cần logits ở VỊ TRÍ CUỐI

        probs = torch.softmax(logits, dim=-1)                       # (1, vocab)

        # --- top-p / nucleus filtering ---
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cum_probs = torch.cumsum(sorted_probs, dim=-1)

        remove_mask = cum_probs > top_p
        remove_mask[..., 1:] = remove_mask[..., :-1].clone()  # dịch phải 1 -> luôn giữ token vừa vượt ngưỡng
        remove_mask[..., 0] = False                            # luôn giữ ít nhất 1 token
        sorted_probs[remove_mask] = 0.0
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

        next_in_sorted = torch.multinomial(sorted_probs, num_samples=1)  # sample theo index đã sort
        next_id = sorted_idx.gather(-1, next_in_sorted)                   # map ngược lại token id gốc

        ids = torch.cat([ids, next_id], dim=1)

    model.train()
    return tokenizer.decode(ids[0])


def chat(model, tokenizer: Tokenizer, device, max_new_tokens: int = 200,
         temperature: float = 0.8, top_p: float = 0.9):
    """
    Lưu ý: đây là base LM char-level train bằng next-token prediction thuần trên Shakespeare,
    KHÔNG phải model đã instruction-tune -> "chat" ở đây nghĩa là đưa 1 đoạn mồi (prompt),
    model tiếp tục viết theo văn phong đã học, không phải hỏi-đáp thật.
    """
    print(f"Gõ 'exit' để thoát. (temperature={temperature}, top_p={top_p})")
    while True:
        prompt = input("You: ")
        if prompt.strip().lower() == "exit":
            break
        output = generate(model, tokenizer, prompt, max_new_tokens, device,
                           temperature=temperature, top_p=top_p)
        continuation = output[len(prompt):]  # chỉ in phần model sinh thêm, bỏ lại prompt gốc
        print("Model:", continuation)

chat(model, tokenizer, device, max_new_tokens=200, temperature=0.8, top_p=0.9)

Gõ 'exit' để thoát. (temperature=0.8, top_p=0.9)
You: hi
Model: s war,
And great to this contain hanging.

DUCHESS OF YORK:
What must I call you my son?

DUKE OF AUMERLE:
What must I pit, the duke dark of an age:
Mastal, he madam; he stand still, and he still
Stil
You: Before we proceed any further, hear me speak.
Model: 
Think that is on mine own to age,
And waked the sea banish'd and their princes,
Which with her conclude.

PARIS:
Worthy may mistress of the abundant of death?

GLOUCESTER:
I pray thee, my lord, to th
You: First Citizen
Model: :
Now it befall his tim against he has ten time
The noble devotion of his country, I will proceed
Wherein myself is and power.

CORIOLANUS:
Hark! what a happy stirres are to the day?

CORIOLANUS:
No, 
You: exit
